# Speech-Based Cognitive Screening — Notebook 02
## The Concept Bottleneck · Does interpretability cost accuracy?

**Team:** A. R. Nitesh Kumar (23BAI0108) · Harshini J. S. (23BAI0166) · Hari Prasath U. (23BAI0038)  


**What this notebook does — the core Deep Learning contribution.**  
Notebook 01 proved the clinical markers separate the groups (baseline AUC ≈ 0.78). This notebook
builds the **concept-bottleneck model** and answers the question the whole project is built around:

> **If we force every prediction to pass through human-readable clinical concepts, how much accuracy
> do we lose compared to an unrestricted black box?**

We train three models on the *same* features with the *same* speaker-independent CV:

| Model | Path | Interpretable? |
|---|---|---|
| **A · Black box** | features → MLP → label | ❌ no |
| **B · Concept bottleneck** | features → concepts → **linear head** → label | ✅ exactly |
| **C · Bottleneck + residual** | B, plus a small penalised shortcut | partly |

Then we show the bottleneck's explanation is **mathematically exact** (contribution = weight × concept),
and that the learned weights are **clinically correct**. No audio, no GPU needed — this runs on the
feature file from Notebook 01.


## 0 · Setup

In [ ]:
!pip install torch scikit-learn pandas pyarrow --quiet
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
torch.manual_seed(0); np.random.seed(0)
print("torch", torch.__version__, "— setup OK")

## 1 · Load the feature table from Notebook 01

Upload **`pitt_cookie_features.csv`** (or `.parquet`) using the file panel on the left, then run
this cell. This is the small table Notebook 01 produced — we never re-parse the transcripts.


In [ ]:
import glob
cand = glob.glob("**/pitt_cookie_features.*", recursive=True)
if not cand:
    try:
        from google.colab import files
        print("Select pitt_cookie_features.csv (or .parquet) …")
        up = files.upload()
        cand = list(up.keys())
    except Exception as e:
        print("upload skipped:", e)
assert cand, "Upload pitt_cookie_features.csv and re-run."
path = [c for c in cand if c.endswith(".parquet")] or cand
path = path[0]
df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
print("loaded", path, "—", df.shape, "\n")
print(df.dx.value_counts())

## 2 · Define the task and the concepts

We use the clean binary task **ProbableAD vs Control**. The **concepts** are the clinical markers —
this exact list is the interpretable layer every prediction must pass through.


In [ ]:
d = df[df.dx.isin(["Control", "ProbableAD"])].copy()
d["label"] = (d.dx == "ProbableAD").astype(int)

CONCEPTS = ["mattr", "pronoun_noun_ratio", "mlu_words", "lexical_density",
            "filled_pause_rate", "speech_rate_wpm", "long_pause_ratio",
            "mean_pause_ms", "ttr", "noun_verb_ratio", "pronoun_ratio"]
k = len(CONCEPTS)

X = d[CONCEPTS].values.astype(np.float32)
y = d["label"].values.astype(np.float32)
groups = d["participant_id"].values
print(f"{len(d)} recordings · {d.participant_id.nunique()} participants · {k} concepts")

## 3 · The three models

The bottleneck's key feature: after predicting the `k` concepts, the final layer is a **single
linear layer**. That linearity is what makes the explanation exact later. The black box has no such
restriction; the residual variant adds a small, penalised shortcut around the concepts so we can
measure whether the concepts are a real constraint.


In [ ]:
class BlackBox(nn.Module):
    """Unrestricted: features -> MLP -> logit."""
    def __init__(self, din):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(din,32), nn.ReLU(),
                                 nn.Linear(32,16), nn.ReLU(), nn.Linear(16,1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

class Bottleneck(nn.Module):
    """features -> predicted concepts -> LINEAR head -> logit.
       Concept head is supervised toward the true concept values."""
    def __init__(self, din, k, residual=False):
        super().__init__()
        self.g    = nn.Sequential(nn.Linear(din,32), nn.ReLU(), nn.Linear(32,k))
        self.head = nn.Linear(k, 1)                     # transparent, linear
        self.residual = residual
        if residual:
            self.r = nn.Sequential(nn.Linear(din,8), nn.ReLU(), nn.Linear(8,1))
    def forward(self, x):
        c = self.g(x)
        logit = self.head(c).squeeze(-1)
        if self.residual:
            logit = logit + 0.5 * self.r(x).squeeze(-1)
        return logit, c

print("models defined")

## 4 · Train & evaluate all three — speaker-independent

Same `StratifiedGroupKFold` as Notebook 01 (no participant across folds; the `assert` proves it).
The bottleneck loss adds concept supervision so the middle layer learns to *be* the clinical
markers. We report **AUC mean ± std** over 5 folds.


In [ ]:
def train_eval(kind, epochs=200, lr=0.01):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    aucs, f1s, accs = [], [], []
    for tr, te in sgkf.split(X, y, groups):
        assert not (set(groups[tr]) & set(groups[te])), "LEAKAGE!"
        imp = SimpleImputer().fit(X[tr]); sc = StandardScaler().fit(imp.transform(X[tr]))
        Xtr = torch.tensor(sc.transform(imp.transform(X[tr])), dtype=torch.float32)
        Xte = torch.tensor(sc.transform(imp.transform(X[te])), dtype=torch.float32)
        ytr = torch.tensor(y[tr])
        Ctr = Xtr.clone()                                # concept targets = standardized concepts

        if kind == "blackbox":   m = BlackBox(k)
        elif kind == "bottleneck": m = Bottleneck(k, k, residual=False)
        else:                    m = Bottleneck(k, k, residual=True)

        opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
        bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()
        for _ in range(epochs):
            m.train(); opt.zero_grad()
            if kind == "blackbox":
                loss = bce(m(Xtr), ytr)
            else:
                logit, c = m(Xtr); loss = bce(logit, ytr) + 0.5 * mse(c, Ctr)
            loss.backward(); opt.step()

        m.eval()
        with torch.no_grad():
            out = m(Xte); logit = out[0] if isinstance(out, tuple) else out
            p = torch.sigmoid(logit).numpy()
        pred = (p >= 0.5).astype(int)
        aucs.append(roc_auc_score(y[te], p)); f1s.append(f1_score(y[te], pred))
        accs.append(accuracy_score(y[te], pred))
    return np.mean(aucs), np.std(aucs), np.mean(f1s), np.mean(accs)

print("Speaker-independent 5-fold CV · ProbableAD vs Control\n")
results = {}
for kind, name in [("blackbox","A · Black box MLP"),
                   ("bottleneck","B · Concept bottleneck"),
                   ("residual","C · Bottleneck + residual")]:
    mu, sd, f1, acc = train_eval(kind)
    results[kind] = mu
    print(f"{name:28s} AUC {mu:.3f} ± {sd:.3f}   F1 {f1:.3f}   Acc {acc:.3f}")

gap = results["blackbox"] - results["bottleneck"]
print(f"\n>>> INTERPRETABILITY COST = {gap:+.3f} AUC "
      f"(black box minus bottleneck) <<<")
print(">>> A small gap means interpretability is nearly free — the headline result. <<<")

## 5 · The payoff — the explanation is *exact*

Because the head is linear, each concept's contribution to a prediction is **exactly**
`weight × concept_value`, and these sum (plus bias) to the logit. This is not an approximation like
SHAP — it is arithmetic. We train one bottleneck on all data and read the weights.

**Watch the signs** — they should match clinical theory: more pauses / more pronouns → AD; richer
vocabulary / longer utterances / faster speech → Control.


In [ ]:
imp = SimpleImputer().fit(X); sc = StandardScaler().fit(imp.transform(X))
Xs = torch.tensor(sc.transform(imp.transform(X)), dtype=torch.float32)
yt = torch.tensor(y)

m = Bottleneck(k, k, residual=False)
opt = torch.optim.Adam(m.parameters(), lr=0.01, weight_decay=1e-4)
bce, mse = nn.BCEWithLogitsLoss(), nn.MSELoss()
for _ in range(300):
    m.train(); opt.zero_grad(); logit, c = m(Xs)
    (bce(logit, yt) + 0.5*mse(c, Xs)).backward(); opt.step()

W = m.head.weight.detach().numpy().ravel()
print("=== Concept weights in the transparent head ===")
print(f"{'concept':22s} {'weight':>8s}   pushes toward")
for i in np.argsort(-np.abs(W)):
    print(f"{CONCEPTS[i]:22s} {W[i]:+8.3f}   {'AD' if W[i]>0 else 'Control'}")

## 6 · Prove exactness on one recording

Contribution decomposition: `Σ (weightⱼ × conceptⱼ) + bias == logit`, to numerical precision. This
is what a clinician sees — the exact reasons for *this* patient's score — and it is the ground truth
we will later use to test whether SHAP and LIME are faithful.


In [ ]:
idx = 0
m.eval()
with torch.no_grad():
    logit, c = m(Xs[idx:idx+1])
c = c.numpy().ravel()
contrib = W * c

print(f"Recording: {d.iloc[idx]['file']}  (true label = {int(y[idx])}: "
      f"{'AD' if y[idx]==1 else 'Control'})\n")
print("Top concept contributions to this decision:")
for i in np.argsort(-np.abs(contrib))[:6]:
    print(f"  {CONCEPTS[i]:22s} contribution {contrib[i]:+.3f}")

total = contrib.sum() + m.head.bias.item()
print(f"\nΣ contributions + bias = {total:+.4f}")
print(f"model logit            = {logit.item():+.4f}")
print(f"EXACT match: {np.isclose(total, logit.item(), atol=1e-4)}  ✅  "
      f"(this is why the explanation is faithful by construction)")

## 7 · What you produced & what's next

- A **three-way comparison** quantifying the interpretability cost — your headline Deep Learning result.
- A **linear concept head** whose explanations are exact, with **clinically correct** weight signs.
- The **ground-truth attributions** needed for the XAI study.

### Next options
1. **XAI faithfulness audit (Notebook 03):** run SHAP and LIME on this bottleneck and measure how far
   they diverge from the exact `weight × concept` truth. This is the Explainable-AI contribution and
   needs no new data.
2. **MCI task:** repeat this notebook with the 42 MCI cases — the harder, more valuable problem.
3. **Audio (Experiment E2):** only now fetch the MP3s and add acoustic concepts (pitch, jitter,
   spectrogram CNN), then re-run this comparison with the enlarged concept set.

> Report every number as **mean ± std over folds/seeds**, never a single run — the small values swing
> a little each time, which is expected.
